# Analyze Account Data Sources

We have three sources for account and account holder data:
1. Direct download of the account data
2. Download through the power BI app
3. Inferred from transaction data

We need to examine what are the differences in different source and which information
we take from which of the sources.

## Packages and options

In [2]:
# add the parent directory to the sys.path
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import pandas as pd

In [3]:
fn_direct = "../data/source/automatic/eutl_accounts.csv"
fn_bi = "../data/source/manual/accounts.xlsx"
fn_trans = "../data/source/automatic/eutl_transactions.csv"

## Get data

Account data from direct download:

In [4]:
df_acc_direct = pd.read_csv(fn_direct).assign(
    account_id=lambda df: df["REGISTRY_CODE"]
    + "_"
    + df["ACCOUNT_IDENTIFIER"].astype(str),
)
map_registry_names = df_acc_direct.set_index("REGISTRY_NAME")["REGISTRY_CODE"].to_dict()
map_registry_names.update({"Switzerland": "CH", "CDM": "CDM"})
df_acc_direct.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47666 entries, 0 to 47665
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   ACCOUNT_IDENTIFIER    47666 non-null  int64 
 1   REGISTRY_CODE         47666 non-null  object
 2   REGISTRY_NAME         47666 non-null  object
 3   ACCOUNT_NAME          47666 non-null  object
 4   ACCOUNT_TYPE          47666 non-null  object
 5   ETS_ACCOUNT_TYPE      28348 non-null  object
 6   FULL_TYPE             47637 non-null  object
 7   OPEN_DATE             47630 non-null  object
 8   END_OF_VALIDITY_DATE  47666 non-null  object
 9   IS_CLOSURE_PENDING    47666 non-null  object
 10  SNAPSHOT_DATE         47666 non-null  object
 11  account_id            47666 non-null  object
dtypes: int64(1), object(11)
memory usage: 4.4+ MB


In [5]:
# df_acc_direct.ACCOUNT_TYPE.unique()

In [6]:
# df_acc_direct.ETS_ACCOUNT_TYPE.unique()

Power BI data

In [7]:
df_acc_bi = (
    pd.read_excel(fn_bi, skipfooter=2)
    .rename(columns={"..1": "registry_id"})
    .drop(columns=".")
    .assign(
        account_id=lambda df: df["registry_id"]
        + "_"
        + df["Account Identifier"].astype(str),
    )
)
df_acc_bi.info()

e:\GIT\eutl_scraper_v2\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47389 entries, 0 to 47388
Data columns (total 15 columns):
 #   Column                                               Non-Null Count  Dtype  
---  ------                                               --------------  -----  
 0   Account Identifier                                   47389 non-null  int64  
 1   National Administrator                               47389 non-null  object 
 2   Account Type                                         47389 non-null  object 
 3   Account Holder Name                                  47389 non-null  object 
 4   Account Name                                         47389 non-null  object 
 5   Installation/Aircraft Operator/Maritime Operator ID  22158 non-null  float64
 6   Company Registration No                              42781 non-null  object 
 7   Main Address Line                                    47380 non-null  object 
 8   City                                                 47380 non-nul

Transaction data

In [8]:
df_acc_trans_in = pd.read_csv("../data/source/automatic/eutl_transactions.csv")

C:\Users\abrel\AppData\Local\Temp\ipykernel_39524\313709080.py:1: DtypeWarning: Columns (20,22,23,24,25,26,27,28,29,47,49,50,51,52,53,54,55,56,60,62,65) have mixed types. Specify dtype option on import or set low_memory=False.
  df_acc_trans_in = pd.read_csv("../data/source/automatic/eutl_transactions.csv")


In [9]:
def assign_account_id(row) -> str:
    if pd.notnull(row["ACCOUNT_IDENTIFIER"]):
        if pd.notnull(row["registry_id"]):
            registry_id = row["registry_id"]
        else:
            registry_id = "UNKNOWN"
        return registry_id + "_" + str(int(row["ACCOUNT_IDENTIFIER"]))


df_acc_trans = df_acc_trans_in.copy()
# extract account involved in transactions
lst_df = []
for prefix in ["TRANSFERRING", "ACQUIRING"]:
    cols = [c for c in df_acc_trans.columns if c.startswith(prefix)]
    df_ = df_acc_trans[cols].copy()
    cols = [c.replace(f"{prefix}_", "") for c in cols]
    df_.columns = cols
    lst_df.append(df_)
df_acc_trans = (
    pd.concat(lst_df, axis=0)
    .drop_duplicates()
    .reset_index(drop=True)
    .assign(
        registry_id=lambda df: df["REGISTRY_NAME"].str.strip().map(map_registry_names),
        account_id=lambda df: df.apply(assign_account_id, axis=1),
    )
)
# df_acc_trans.info()

## How do direct downloads compare against power bi accounts?

Overall we observe:

1. Only direct downloads provide information on dates
2. Only the PowerBi downloads provide information on the account holder 

In [10]:
acc_only_direct = set(df_acc_direct["account_id"]) - set(df_acc_bi["account_id"])
acc_only_bi = set(df_acc_bi["account_id"]) - set(df_acc_direct["account_id"])
acc_only_in_trans = set(df_acc_trans["account_id"]) - set(df_acc_direct["account_id"])
print(f"Accounts only in direct download: {len(acc_only_direct)}")
print(f"Accounts only in BI download: {len(acc_only_bi)}")
print(f"Accounts only in transactions download: {len(acc_only_in_trans)}")

Accounts only in direct download: 277
Accounts only in BI download: 0
Accounts only in transactions download: 744


### Account types

#### Power BI

Account types are aggregated by groups and more disaggregated in the Power BI app. 
However, the information is somewhat mixed across columns. The pattern seems to be:

1. Holding accounts: 
   1. ACCOUNT_TYPE: is Holding Account
   2. ETS_ACCOUNT_TYPE: provides the detailed account
   3. FULL_TYPE: Repeats ETS_ACCOUNT_TYPE (Not for all)
2. Other account:
   1. ETS_ACCOUNT_TYPE: missing
   2. FULL_TYPE: Is more detailed

In [11]:
df_ = df_acc_direct[["ACCOUNT_TYPE", "ETS_ACCOUNT_TYPE", "FULL_TYPE"]].drop_duplicates()
print("Differences in ACCOUNT_TYPE vs FULL_TYPE:")
df_[
    (df_["ACCOUNT_TYPE"].str.strip() != df_["FULL_TYPE"].str.strip())
    & (df_["ACCOUNT_TYPE"] != "Holding Account")
].drop_duplicates()

Differences in ACCOUNT_TYPE vs FULL_TYPE:


,ACCOUNT_TYPE,ETS_ACCOUNT_TYPE,FULL_TYPE
0,Operator Holding Account,NaN,Former Operator Holding Account
15671,Non-Kyoto Account Type,Verifier Account,Verifier Account


In [12]:
# df_acc_bi[["Account Type"]].drop_duplicates()

#### Differences Power Bi and direct


We need to check the differences in the account types:

As we can see, that the differences mainly stem from the accounts that are missing
in the BI downloads. However, we have some account types that are only identified by
the direct downloads (our lovely Credit Accounts... that at least exist in the 
new version of the EC data...).

In [13]:
df_acc = df_acc_direct.merge(df_acc_bi, on="account_id", how="outer")
direct, bi = "FULL_TYPE", "Account Type"
df_ = df_acc[(df_acc[direct].str.strip() != df_acc[bi].str.strip())]
print(f"{len(df_)} differing account types between direct downloads and Power BI")
df_ = df_[[direct, bi]].sort_values(by=direct).drop_duplicates()
df_

281 differing account types between direct downloads and Power BI


,FULL_TYPE,Account Type
14024,AEA Deletion Account,NaN
14023,AEA Total quantity Account,NaN
14128,ESD Compliance Account,NaN
2488,Former Operator Holding Account,NaN
36729,Party Holding Account,NaN
2487,Person Account in National Registry,NaN
2503,Retirement Account,NaN
2502,Voluntary Cancellation Account (Type 3),NaN
680,NaN,NaN
18378,NaN,International Credit Account


#### Account Names

Comparing names between direct and power BI downloads, it seems that the BI downloads
(the column Account Name) have been normalized. It also appears that the BI downloads 
have more missing values. We therefore prefer the direct downloads.

In [14]:
df_acc = df_acc_direct.merge(df_acc_bi, on="account_id", how="outer")
to_check = {
    "Account Name": "ACCOUNT_NAME",
}

for right, left in to_check.items():
    df_ = df_acc[(df_acc[left].str.strip() != df_acc[right].str.strip())]
    if not df_.empty:
        df_ = df_[[left, right]].sort_values(by=left).drop_duplicates()
        break
df_acc[[left, right]].drop_duplicates().sort_values(by=left)
df_.info()
df_

<class 'pandas.core.frame.DataFrame'>
Index: 542 entries, 13293 to 47607
Data columns (total 2 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   ACCOUNT_NAME  542 non-null    object
 1   Account Name  265 non-null    object
dtypes: object(2)
memory usage: 12.7+ KB


,ACCOUNT_NAME,Account Name
13293,A/S GLOBAL RISK MANAGEMENT LTD. HOLDING,A/S Global Risk Management Ltd. Holding
28081,A2A S.p.A.,A2A S.P.A.
34774,A2A Trading,a2a Trading
22613,ABN AMRO BANK N.v.,ABN AMRO BANK N.V.
39253,ABN AMRO Bank N.V.,ABN AMRO BANK N.V.
...,...,...
33235,stabilimento di Brindisi,Stabilimento di Brindisi
11877,ubr logistik,UBR Logistik
34790,veronagest,Veronagest
5641,vertus energiehandel gmbh,Vertus Energiehandel Gmbh


## How do transaction data compare against direct downloads

There are plenty of accounts that are only in the direct downloads but not in the transaction data. This is natural
as there are accounts that do not transfer allowances. There are also accounts, that are in the transaction data
but not in the downloaded account table. 

Closer inspection reveals that these are mainly accounts that are not registered 
in the European system but involved in transactions. These are accounts are pretty
raw in the sense that they do not provide account names or any information.

In [19]:
acc_only_direct = set(df_acc_direct["account_id"]) - set(df_acc_trans["account_id"])
acc_only_in_trans = set(df_acc_trans["account_id"]) - set(df_acc_direct["account_id"])
print(f"Accounts only in direct download: {len(acc_only_direct)}")
print(f"Accounts only in transactions download: {len(acc_only_in_trans)}")

Accounts only in direct download: 13355
Accounts only in transactions download: 744


In [18]:
df_ = df_acc_trans[df_acc_trans["account_id"].isin(acc_only_in_trans)].sort_values(
    by="account_id"
)
df_.info()
df_

<class 'pandas.core.frame.DataFrame'>
Index: 790 entries, 13493 to 34370
Data columns (total 29 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   REGISTRY_NAME                               790 non-null    object 
 1   ACCOUNT_TYPE1                               760 non-null    float64
 2   ACCOUNT_TYPE2                               790 non-null    object 
 3   ACCOUNT_TYPE3                               790 non-null    object 
 4   ACCOUNT_OPEN_DT                             0 non-null      object 
 5   ACCOUNT_END_OF_VALIDITY                     0 non-null      object 
 6   ACCOUNT_NAME                                0 non-null      object 
 7   ACCOUNT_IDENTIFIER                          760 non-null    float64
 8   ACCOUNT_HOLDER                              0 non-null      object 
 9   ACCOUNT_HOLDER_ADDRESS1                     0 non-null      object 
 10  ACCOUNT_HOLDE

,REGISTRY_NAME,ACCOUNT_TYPE1,ACCOUNT_TYPE2,ACCOUNT_TYPE3,ACCOUNT_OPEN_DT,ACCOUNT_END_OF_VALIDITY,ACCOUNT_NAME,ACCOUNT_IDENTIFIER,ACCOUNT_HOLDER,ACCOUNT_HOLDER_ADDRESS1,...,INSTALLATION_PARENT_COMPANY,INSTALLATION_SUBSIDIARY_COMPANY,INSTALLATION_EPER_IDENTIFICATION,INSTALLATION_CITY,INSTALLATION_POSTAL_CODE,INSTALLATION_ADDRESS1,INSTALLATION_ADDRESS2,INSTALLATION_MAIN_ACTIVITY,registry_id,account_id
13493,CDM,110.0,-,-,NaN,NaN,NaN,1000.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_1000
22157,CDM,100.0,-,-,NaN,NaN,NaN,1004.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_1004
27979,CDM,100.0,-,-,NaN,NaN,NaN,2004.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_2004
22201,CDM,100.0,-,-,NaN,NaN,NaN,2005.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_2005
22289,CDM,100.0,-,-,NaN,NaN,NaN,2006.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,CDM,CDM_2006
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26923,European Commission,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,EU,None
27354,Norway,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,NO,None
30322,Estonia,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,EE,None
31437,Liechtenstein,NaN,-,-,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-,LI,None


## Unify accounts

To get a single account table we follow the following strategy:

1. The direct download is our base table
2. Account Types:
   1. If ETS_ACCOUNT_TYPE is missing, we ingest it from the power bi table (to the FULL_TYPE column)
   2. SUPPLEMENTARY_TRANSACTION_TYPE:
      1. ETS_ACCOUNT_TYPE is the base column
      2. Missings are filled from "FULL_TYPE"
      3. Remaining missings are filled from power bi table column "Account Type"
3. Account Name:
   1. direct download ACCOUNT_NAME is the base column
4. Installation Mapping: 
   1. The mapping is already given in the installation data. So for the first moment we ignore.
   2. Later we need to check for consistency and whether there is additional information
5. Account Holder Mapping


In [ ]:
from pathlib import Path


def unify_accounts(
    df_direct: pd.DataFrame,
    df_bi: pd.DataFrame,
    df_trans: pd.DataFrame,
    fn_out: str | Path | None = None,
) -> pd.DataFrame:
    """Unify account data from different sources into a single DataFrame.

    Args:
        df_direct (pd.DataFrame): DataFrame containing account data from direct download.
        df_bi (pd.DataFrame): DataFrame containing account data from Power BI.
        df_trans (pd.DataFrame): DataFrame containing account data from transactions.
        fn_out (str | Path | None): Optional file path to save the unified DataFrame as CSV.

    Returns:
        pd.DataFrame: Unified DataFrame containing account data from all sources.
    """